In [1]:
import numpy as np
import pandas as pd
import warnings
import os
import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
def get_dataset(dataset:str)->str:
    try:
        return pd.read_csv(os.path.join('..','..','data','raw', f'{dataset}.csv'))
    except Exception as e:
        warnings.warn(f'"{dataset}" dataset not present.')
        return None

In [3]:
def check_is_unique(dataframe,column_names):
    '''
    print that the given columns is unique or not.
    parameters:
    dataframe: pandas dataframe
    column_name : name of the column which want to check.
    '''
    if  type(column_names) == str:
        print(f"'{column_names}' is a unique in table ? { dataframe[column_names].is_unique}")
        return; 
    else:
        for column_name in column_names:
            print(f"'{column_name}' is a unique in table ? { dataframe[column_name].is_unique}")


In [4]:
# Load the dataset
df_customers = get_dataset('customers')
df_items  = get_dataset('order_items')
df_geolocation = get_dataset('geolocation')
df_sellers = get_dataset('sellers')
df_orders = get_dataset('orders')
df_reviews = get_dataset('order_reviews')
df_product = get_dataset('products')
df_product_category_eng =  get_dataset('product_category_name_translation') 

df_payment = get_dataset('order_payments')

df_leads_closed = get_dataset('leads_closed')


In [5]:
df_geolocation.head()

,geolocation_zip_code_prefix,geolocation_lat,geolocation_lng,geolocation_city,geolocation_state
0,1001,-23.551427,-46.634074,sao paulo,SP
1,1001,-23.551337,-46.634027,sao paulo,SP
2,1001,-23.550642,-46.634410,sao paulo,SP
3,1001,-23.550498,-46.634338,sao paulo,SP
4,1001,-23.550263,-46.634196,são paulo,SP


#### Merge

In [6]:
df_unique_geoloation = df_geolocation.groupby('geolocation_zip_code_prefix').agg(latitude = ('geolocation_lat','median'),longitude = ('geolocation_lng','median')).reset_index()

In [7]:
# Merge Customer and geolocation table
df_customer_geo = df_customers.merge(right = df_unique_geoloation, left_on ='customer_zip_code_prefix',right_on='geolocation_zip_code_prefix',how='left')
df_customer_geo.drop(columns=['geolocation_zip_code_prefix'],inplace=True)
df_customer_geo.rename(columns={'latitude':'customer_latitude','longitude':'customer_longitude'},inplace=True)


In [8]:
# Merge Seller and geolocation table
df_seller_geo = df_sellers.merge(right = df_unique_geoloation, left_on ='seller_zip_code_prefix',right_on='geolocation_zip_code_prefix',how='left')
df_seller_geo.drop(columns=['geolocation_zip_code_prefix'],inplace=True)
df_seller_geo.rename(columns={'latitude':'seller_latitude','longitude':'seller_longitude'}, inplace=True)

In [9]:
# merge product and product category english translation
df_prduct_decs_eng = df_product.merge(df_product_category_eng,on='product_category_name',how='left',validate='many_to_one')


In [10]:
# order_items and df_prduct_decs_eng
df_items_product = df_items.merge(right=df_prduct_decs_eng,on='product_id',how='left',validate='many_to_one')

In [11]:
# join orders and payment table
df_order_payment =  df_orders.merge(df_payment, on='order_id', how='left', validate='one_to_one')


In [12]:
# merge items with seller lookup

df_items_seller =  df_items_product.merge(df_seller_geo,on='seller_id',how='left',validate='many_to_one')

In [13]:
# merge customer info with order talbe

df_order_payment_custGeo =  df_order_payment.merge(df_customer_geo,on='customer_id',how='left',validate='many_to_one')


In [14]:
# join order table with order_items table

df_master = df_order_payment_custGeo.merge(df_items_seller,on='order_id',how='left',validate='one_to_many')

In [18]:
df_master = df_master[df_master['order_status']=='delivered']

In [19]:
# save data....
df_master.to_csv(os.path.join('..','..','data','processed','merged_data.csv'),index=False)